# qwen21-nsfw-pose · 非人物题材示例（风景 / 动物 / 表情包 / 改图 / 批量）

> 先在「更改运行时类型」选 **T4 GPU**，并确保已按 `colab_t4_quick.ipynb` 跑通过一次（底模、ComfyUI 已就绪）。
>
> 这些示例都**不使用 NSFW LoRA**：`--preset` 会自动关闭 LoRA 并选好画幅，且**必须自己写提示词**
> （默认样例提示词是作者的人像，不适用于这些题材）。
>
> ⚠️ 非人物题材、文字渲染和改图在 T4 上**均未实测**，效果请以实际出图为准。
>
> 提示词里避免使用英文双引号 `"`，否则会破坏 shell 命令；需要引号请用单引号。
>
> 每个示例 cell 都是独立的：想试哪个就只运行哪个，运行后用最后一个 cell 查看结果。

In [ ]:
# ===== 准备：挂载 Drive + 更新仓库 =====
REPO_URL = "https://github.com/ldybyz/qwen21-nsfw-pose-skill.git"
BRANCH = "feat/colab-t4-guide"  # 合并进 main 后可改回 main
POSELAB = "/content/qwen21-nsfw-pose-skill"

from google.colab import drive
drive.mount("/content/drive")

!test -d {POSELAB} || git clone -b {BRANCH} {REPO_URL} {POSELAB}
!cd {POSELAB} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}

In [ ]:
# ===== 示例 1：风景（preset=landscape，1024x576，无 LoRA）=====
PROMPT = (
    "A misty mountain lake at sunrise, pine forest on the shore, "
    "golden light breaking through low clouds, calm water reflection, "
    "ultra detailed landscape photograph, wide angle, natural colors"
)
!python {POSELAB}/scripts/colab_quickstart.py --preset landscape --prompt "{PROMPT}" --seed 1 --skip-doctor

In [ ]:
# ===== 示例 2：动物（preset=animal，1024x768，无 LoRA）=====
PROMPT = (
    "A golden retriever puppy sitting on a wooden porch, morning sunlight, "
    "shallow depth of field, detailed fur, warm tones, "
    "professional pet photography"
)
!python {POSELAB}/scripts/colab_quickstart.py --preset animal --prompt "{PROMPT}" --seed 1 --skip-doctor

In [ ]:
# ===== 示例 3：表情包 / 带文字的图（preset=sticker，1024x1024，无 LoRA）=====
# 文字渲染效果未验证；文字尽量短，并在提示词里明确写出要显示的内容。不支持透明背景。
PROMPT = (
    "A cute cartoon cat sticker, thick white outline, flat colors, "
    "the cat is holding a sign that says 'OK', plain light background, "
    "simple clean vector style"
)
!python {POSELAB}/scripts/colab_quickstart.py --preset sticker --prompt "{PROMPT}" --seed 1 --skip-doctor

In [ ]:
# ===== 示例 4：改已有的图（非人物，例如换背景 / 换风格）=====
# 先把原图上传到 Drive 的 qwen21_pose/inputs/。改图默认不带 LoRA、cfg 3、20 步。
# 指令用英文；一次只改一件事，并写明要保留的部分。
INPUT_IMAGE = "/content/drive/MyDrive/qwen21_pose/inputs/my_pet.png"
INSTRUCTION = "change the background to a snowy forest, keep the animal and its pose unchanged"
!python {POSELAB}/scripts/colab_quickstart.py --mode edit --input "{INPUT_IMAGE}" --prompt "{INSTRUCTION}" --skip-doctor

In [ ]:
# ===== 示例 5：同一提示词批量出多张（换种子）=====
# 脚本一次只出 1 张，这里循环调用。每张约 5 分钟（T4，1024x768，20 步，参考值），请按需调整数量。
import subprocess

PROMPT = (
    "A red fox walking through a snowy forest at dawn, soft backlight, "
    "detailed fur, shallow depth of field, wildlife photography"
)
for seed in (1, 2, 3):
    subprocess.run(
        ["python", f"{POSELAB}/scripts/colab_quickstart.py", "--preset", "animal",
         "--prompt", PROMPT, "--seed", str(seed), "--skip-doctor"],
        check=False,
    )

In [ ]:
# ===== 查看最近生成的几张图 =====
import glob, os
from IPython.display import Image, display

SHOW_LAST = 3
imgs = sorted(glob.glob(f"{POSELAB}/outputs/colab*.png"), key=os.path.getmtime)
if not imgs:
    print("还没有输出图，先运行上面的示例")
for p in imgs[-SHOW_LAST:]:
    print(p)
    display(Image(p, width=512))